# Comparando Planos de Pagamento de Empréstimo Estudantil com PROC LOAN

## Resumo Executivo

Um setor de assistência estudantil avalia como uma turma de formandos deve pagar um saldo representativo de $27,500 comparando quatro estruturas de pagamento — um plano federal padrão de taxa fixa, um refinanciamento privado com taxa de abertura, um empréstimo de taxa variável com teto (ARM) e uma redução de taxa (buydown) patrocinada pelo empregador — usando `PROC LOAN`. Ao longo de um prazo de 120 meses, as quatro ofertas se alinham claramente quanto ao pagamento mensal e aos juros totais em suas taxas iniciais cotadas: o plano federal padrão custa mais em juros (**$10,021.22** a 6.53%, parcela **$312.68**), o refinanciamento privado reduz a taxa mas adiciona uma taxa de $412.50 (**$9,120.20** a 5.99%, parcela **$305.17**), e o ARM com cotação mais baixa (**$7,099.76** a 4.75%, parcela **$288.33**) e a redução de taxa do empregador (**$6,700.67** a 4.50%, parcela **$285.01**) apresentam as contas de juros mais baixas. Um bloco `COMPARE` então relata os juros acumulados, o principal e o saldo devedor de cada plano aos 36, 60 e 120 meses, mostrando o quanto cada empréstimo já foi amortizado nos pontos em que um mutuário tem mais chance de refinanciar ou quitar.

## Fontes de Dados

| Conjunto de Dados | Linhas | Descrição | Variáveis-Chave |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Perfis sintéticos de empréstimo de uma turma de formandos, gerados inline com `call streaminit(20260531)` e `rand('uniform')`. Usados para motivar prazos de empréstimo realistas para a comparação. | `student_id` (1001-1040), `balance` (principal na formatura; este sorteio abrange $11,800-$47,750, média $30,771), `apr` (taxa nominal anual de 4.10%-9.10%, média 6.50%), `term` (120 ou 180 meses, média 144), `origination_pct` (taxa de 1.0%-2.0%, média 1.50%) |

O mutuário representativo analisado com `PROC LOAN` (valor $27,500, prazo de 120 meses, início em julho de 2026) fica próximo à faixa média-baixa da distribuição de saldo e taxa desta turma; nenhum dado externo ou de rede é usado. A turma existe para motivar prazos de empréstimo plausíveis — a comparação direta é feita no único empréstimo representativo.

# Comparando Planos de Pagamento de Empréstimo Estudantil com PROC LOAN

Quando os estudantes se formam, um setor de assistência estudantil precisa ajudá-los a escolher entre ofertas de pagamento concorrentes. `PROC LOAN` (SAS/ETS) foi criado exatamente para isso: ele amortiza empréstimos de taxa fixa, taxa ajustável (ARM) e redução de taxa (buydown), e depois os compara lado a lado quanto a parcela, juros totais e progresso de amortização.

Neste notebook nós:

1. Geramos uma turma sintética de formandos para estabelecer prazos de empréstimo realistas.
2. Resumimos a turma com `PROC MEANS`.
3. Construímos quatro planos de pagamento para um saldo representativo de $27,500 e os comparamos com `PROC LOAN` (FIXED / ARM / BUYDOWN + COMPARE).
4. Executamos novamente, isoladamente, o plano fixo recomendado para confirmar sua economia individual.

Tudo é executado localmente com dados sintéticos inline — sem arquivos externos ou acesso à rede.

## Etapa 1 — Gerar uma turma sintética de formandos

Simulamos 40 mutuários. Cada um sorteia um saldo principal na formatura, uma taxa anual (APR) vagamente ligada à qualidade de crédito, um prazo padrão de pagamento (10 ou 15 anos) e um percentual de taxa de abertura. A semente (seed) torna os dados reproduzíveis.

In [1]:
DADOS borrowers;
   CHAMAR streaminit(20260531);
   COMPRIMENTO plan $ 28;
   FAZER student_id = 1001 ATÉ 1040;
      /* Saldo do principal na formatura: $9.500 - $48.000 */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Taxa anual nominal (APR) por nível de crédito: 4,0% - 9,5% */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Prazo padrão de pagamento: 120 ou 180 meses */
      SE rand('uniform') < 0.6 ENTÃO term = 120;
      SENÃO term = 180;
      /* Taxa de abertura como percentual do principal: 1,0% - 2,0% */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      SAÍDA;
   FIM;
EXECUTAR;


NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Etapa 2 — Perfil da turma

Antes de modelar empréstimos individuais, examinamos a distribuição de saldos, taxas e prazos. Isso nos mostra como é um mutuário *representativo* — a base para a comparação direta que se segue.

In [2]:
PROCEDIMENTO MÉDIAS DADOS=borrowers n mean MIN MAX maxdec=2;
   VARIÁVEL balance apr term origination_pct;
   RÓTULO balance="Saldo do Empréstimo (USD)"
          apr="Taxa Anual (%)"
          term="Prazo (meses)"
          origination_pct="Taxa de Abertura (%)";
EXECUTAR;

                                                  The MEANS Procedure

 Variable         Label                             N           Mean     Minimum     Maximum
 -------------------------------------------------------------------------------------------
 balance          Saldo do Empréstimo (USD)        40       30771.25    11800.00    47750.00
 apr              Taxa Anual (%)                   40           6.50        4.10        9.10
 term             Prazo (meses)                    40         144.00      120.00      180.00
 origination_pct  Taxa de Abertura (%)             40           1.50        1.00        2.00
 -------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Etapa 3 — Comparar quatro planos de pagamento para um mutuário representativo

Usando um saldo representativo de $27,500 ao longo de um prazo de 120 meses (10 anos) com início em julho de 2026, alinhamos quatro ofertas realistas:

- **Federal Direto Não Subsidiado (Padrão)** — um empréstimo simples de taxa fixa a 6.53%.
- **Refinanciamento Privado (com Taxa)** — uma taxa fixa menor de 5.99%, mas com um custo de abertura de $412.50 (`INIT=`).
- **ARM Privado de Taxa Variável** — um empréstimo ajustável de 4.75% com teto de 1% por ajuste / 5% vitalício (`CAPS=`), uma `MAXRATE=` de 9.75%, `ADJUSTFREQ=` anual, e a palavra-chave `WORSTCASE`.
- **Redução de Taxa do Empregador 2-1 (Buydown)** — um início subsidiado de 4.50% que, no cronograma cotado, sobe via `BDRATES=` para 5.50% no ano 2 e 6.50% depois.

A instrução `COMPARE` solicita a visão cruzada entre empréstimos aos 36, 60 e 120 meses, com uma `TAXRATE=` de 22% e uma taxa mínima atrativa de retorno de 4% (`MARR=`); `OUTSUM=` e `OUTCOMP=` capturam o resumo por empréstimo e as linhas de comparação. A listagem abaixo relata, para cada plano e cada ponto de verificação, os **juros acumulados pagos, o principal acumulado quitado e o saldo devedor** — uma visão clara do progresso de amortização entre os candidatos.

> **Nota sobre ajustes de taxa.** O `PROC LOAN` do Jenner amortiza cada plano em sua taxa **inicial** cotada; assim, as configurações `CAPS`/`MAXRATE`/`WORSTCASE` do ARM e os degraus `BDRATES` do buydown são ecoados na listagem como termos do empréstimo, mas **não** são aplicados aos cálculos de parcela — os valores do ARM e do buydown abaixo refletem suas taxas iniciais de 4.75% e 4.50% mantidas fixas por todo o prazo. Trate esses dois totais como custos de melhor cenário (taxa inicial), não como tetos de pior cenário.

In [3]:
PROCEDIMENTO loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         label="Federal Direto Não Subsidiado (Padrão)";

   fixed rate=5.99 init=412.50
         label="Refinanciamento Privado (com Taxa)";

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       label="ARM Privado de Taxa Variável (Pior Cenário)";

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           label="Redução de Taxa do Empregador 2-1, depois 6,5%";

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 outcomp=plan_cmp;
EXECUTAR;

                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Federal Direto Não Subsidiado (Padrão)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Refinanciamento Privado (com Taxa)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: ARM Privado de Taxa Variável (Pior Cenário)
    Loan Type:                   Adjustable Rate
    Amount (P


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Etapa 4 — Executar novamente, isoladamente, o plano fixo recomendado

Para o mutuário que valoriza a previsibilidade da parcela, o plano federal padrão de taxa fixa é a opção segura padrão. Executamos novamente em isolamento para confirmar sua economia principal: a mesma parcela mensal de **$312.68**, total pago de **$37,521.22** e juros totais de **$10,021.22** vistos na comparação entre os quatro planos, agora apresentados como um resumo de empréstimo autônomo.

In [4]:
PROCEDIMENTO loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed label="Federal Direto Não Subsidiado (Padrão)";
EXECUTAR;

                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Federal Direto Não Subsidiado (Padrão)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Federal Direto Não Subsidiado (Padrão)
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         4         2103


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Interpretando os resultados

Todos os quatro planos amortizam totalmente o mesmo principal de $27,500 ao longo de 120 meses (o saldo devedor de cada plano chega a $0.00 no mês 120 na tabela COMPARE), então a decisão gira em torno da **parcela mensal** e dos **juros totais na taxa cotada**:

| Plano | Taxa | Parcela | Juros totais | Observações |
|------|------|---------|----------------|-------|
| Federal Direto Padrão | 6.53% | $312.68 | $10,021.22 | Sem taxa; proteções mais fortes ao mutuário |
| Refinanciamento Privado | 5.99% | $305.17 | $9,120.20 | Taxa de abertura de $412.50 |
| ARM Privado de Taxa Variável | 4.75% (inicial) | $288.33 | $7,099.76 | Taxa pode subir; valor é o custo na taxa inicial |
| Redução de Taxa do Empregador 2-1 | 4.50% (inicial) | $285.01 | $6,700.67 | Depende da continuidade do emprego |

- O plano **federal padrão** é o mais caro em juros ($10,021.22), mas oferece uma parcela fixa e previsível de $312.68, sem taxa.
- O **refinanciamento privado** reduz a taxa para 5.99% ($9,120.20 de juros, $901 a menos que o plano federal), mas cobra uma taxa de abertura de $412.50, então sua vantagem líquida sobre o plano federal é de aproximadamente $488 de juros menos a taxa — significativa apenas se o empréstimo durar tempo suficiente para não ser refinanciado antes.
- O **ARM** e o **buydown** mostram os juros mais baixos aqui ($7,099.76 e $6,700.67) puramente porque suas taxas **iniciais** estão bem abaixo das ofertas fixas. Como observado na Etapa 3, o Jenner mantém essas taxas iniciais fixas, então esses são valores de melhor cenário: um ARM real que se ajusta para cima — ou um buydown que perde o subsídio do empregador — resultaria em valores mais altos, e a parcela não é garantida.

A tabela `COMPARE` refina isso mostrando a rapidez com que cada plano amortiza. Aos 36 meses, o plano federal pagou $4,792.27 de juros e quitou $6,464.10 de principal (saldo $21,035.90), enquanto o buydown pagou apenas $3,263.97 de juros e quitou $6,996.24 de principal (saldo $20,503.76) — os planos de taxa mais baixa custam menos juros e reduzem o principal mais rápido nos primeiros três anos. Para um formando avesso a risco, a certeza de taxa do plano federal padrão costuma justificar seus juros mais altos; para um mutuário confiante em um emprego estável e duradouro, o início subsidiado do buydown entrega a maior economia entre as opções que de fato fixam sua taxa.